In [2]:
import concurrent.futures
from openai import OpenAI

client = OpenAI(base_url="http://localhost:8001/v1", api_key="not-needed")

# 待处理的文章列表
articles = [
    "人工智能正在改变医疗行业，通过深度学习算法，AI可以辅助医生进行影像诊断...",
    "区块链技术在供应链管理中的应用越来越广泛，它提供了透明的追溯机制...",
    "量子计算的突破使得传统加密算法面临挑战，后量子密码学成为研究热点...",
    # ... 更多文章
]

# 共享的评分 Prompt 模板（这部分的 KV Cache 只计算一次）
SYSTEM_PROMPT = """你是一个专业的文章评审员。请从以下三个维度评估文章：
1. 清晰度（1-10分）：逻辑是否清晰，表达是否流畅
2. 深度（1-10分）：是否有深入分析，不是泛泛而谈
3. 实用性（1-10分）：对读者是否有实际参考价值
请用 JSON 格式返回评分结果。"""

def score_article(article):
    response = client.chat.completions.create(
        model="Qwen3-0.6B",
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f"请评估以下文章：\n\n{article}"}
        ],
        max_tokens=1000,
        temperature=0.3
    )
    return response.choices[0].message.content

# 并发发送请求（SGLang 自动识别共享前缀并复用 KV Cache）
with concurrent.futures.ThreadPoolExecutor(max_workers=8) as executor:
    results = list(executor.map(score_article, articles))

for i, result in enumerate(results):
    print(f"文章 {i+1} 评分: {result}")

文章 1 评分: <think>
好的，我现在需要评估用户提供的文章的三个维度：清晰度、深度和实用性。首先，我得仔细阅读用户给的文章内容，确保自己理解每个部分。

用户的文章开头提到人工智能正在改变医疗行业，接着用深度学习算法作为例子，说明AI可以辅助医生进行影像诊断。看起来结构比较清晰，先点明主题，然后给出具体应用，可能还有后续内容，但用户没有提供更多信息。所以清晰度可能得打9分，因为逻辑明确，但可能缺乏细节。

接下来是深度。用户的文章只是简单陈述了AI在医疗中的应用，没有深入分析具体的技术细节，比如深度学习算法的工作原理，或者实际案例的具体数据。因此深度可能在7分左右，因为没有深入探讨这些方面。

然后是实用性。文章提到了AI辅助诊断，但没有讨论其实际效果，比如准确率、成本效益或对医生的影响。实用性可能得8分，因为虽然提到了应用，但缺乏具体信息，无法帮助读者做出决策。

综合来看，这三个维度的评分可能分别是9、7、8分。不过需要确认是否有遗漏的信息，比如是否提到了其他方面，或者是否有实际案例支持。如果用户的文章确实没有深入分析，那么深度和实用性可能需要调整。但根据现有信息，应该按照上述评分来评估。
</think>

```json
[
  {
    "clearness": 9,
    "depth": 7,
    "practicality": 8
  }
]
```
文章 2 评分: <think>
好的，我现在需要评估用户提供的文章的三个维度：清晰度、深度和实用性。首先，我得仔细阅读用户给的文章内容，理解其内容和结构。

用户的文章开头提到区块链技术在供应链管理中的应用越来越广泛，并指出它提供了透明的追溯机制。看起来文章的结构比较简单，先陈述现象，然后说明其优势。接下来可能需要检查是否有足够的细节或深入分析。

关于清晰度，文章的结构是陈述事实，没有使用复杂的术语，但可能还是可以提升流畅度。比如，是否每个句子之间有逻辑连接，是否信息传达明确。可能需要分点或使用连接词来增强流畅性。

深度方面，文章提到了区块链在供应链中的应用，但没有展开具体的技术细节或案例，只提到“透明的追溯机制”。这可能显得不够深入，因为可能读者需要了解区块链如何具体实现这些机制，或者其对供应链的影响。因此，深度可能在1-2分之间，因为分析不够深入。

实用性方面，文章提到了应